In [197]:
#decision tree

In [198]:
import numpy as np
import pandas as pd


In [199]:
data=pd.DataFrame([[1,1,'yes'],
                   [1,1,'yes'],
                   [1,0,'no'],
                   [0,1,'no'],
                   [0,1,'no']],columns=['no surfacing','flippers','result'])

In [200]:
data

,no surfacing,flippers,result
0,1,1,yes
1,1,1,yes
2,1,0,no
3,0,1,no
4,0,1,no


In [201]:
#ID3

In [202]:
def information_entropy(data,P=None,N=None):
    data=data.set_index('result')
    length=len(data)
    
    try:
        positive=len(data.loc[P])
    except:
        positive=0
    try:
        nagetive=len(data.loc[N])
    except:
        nagetive=0    #in fact,the classify should stop
    ent=-(positive/length*np.log2(positive/length+1e-9)+nagetive/length*np.log2(nagetive/length+1e-9))
    return ent

    
    

In [203]:
information_entropy(data,P='yes',N='no')

np.float64(0.9709505915692787)

In [204]:
#add class and the information_entropy added

#data.iloc[0,2]='maybe'
#information_entropy(data)

#[out]np.float64(1.0575424730244998)

In [205]:
def find_bestsonclass(data):
    data_columnclass=data.columns[:-1]
    data_len=len(data_columnclass)
    ent=np.zeros(data_len)
    bestsonclass=''
    min_ent=None
    _=0

    for i in data_columnclass:
        data_choose=data.set_index(i)
        sonclass_name=list(set(data_choose.index))
        sonclass_number=len(sonclass_name)

        for j in range(sonclass_number):
            sonclass=data_choose.loc[[sonclass_name[j]]] #use[[ ]] in case of becoming a Series
            each_sonclass_len=len(sonclass)
            ent[_]+=each_sonclass_len/data_len*information_entropy(sonclass,P='yes',N='no')    #信息增益(information gain)

        if min_ent==None:
            min_ent=ent[_]
            bestsonclass=i
        elif min_ent>ent[_]:
            min_ent=ent[_]
            bestsonclass=i
    
        _+=1

    print(ent,min_ent,bestsonclass)
    return bestsonclass

In [206]:
find_bestsonclass(data)

[1.37744375 1.99999999] 1.3774437453109543 no surfacing


'no surfacing'

In [207]:
def split_data(data,name,value):
    data_split=data.set_index(name)
    #sonclass_values=list(set(data.index)).sort()
    return data_split.loc[value]

In [284]:
def get_major_result(data):
    #s=pd.Series(0)
    #class_values=data['result'].unique()
    #data_tem=data.set_index('result')
    #for i in class_values:
    #    class_length=len(data_tem[i])
    #    value_length=pd.Series((i,class_length))
    #    s=pd.concat([s,value_length])
    #major_result=s[s==s.max()].index[0]
    #return major_result    
#第一次的又臭又长。应该注意是在已有条件下执行该函数
    data_tem=data.value_counts().sort_values()
    major_result=data_tem.index[0][-1]
    return major_result

In [285]:
lis=[]
def creat_tree(data):
    global lis
    lis.append(1)
    if len(data[['result']].value_counts())==1:
        if len(data[['result']])==1:  #a Series
            return data['result']    #a value
        else:
            return data['result'].iloc[0]    #a Series and get the first value
        
    elif len(data.columns)==1 or len(data.value_counts())==1:
        return get_major_result(data)
    #elif pd.isna(data.loc[:,:-2].values).all:   #NaN is not equal anything.Include itself.
    #    pass

    elif len(lis)>10:
        return -999
    else:
        node=find_bestsonclass(data)
        mytree={node:{}}
        sonclass_values=data[node].unique()
        for sonclass in sonclass_values:
            sonclass=sonclass.item()    #对数值和字符串，结果不同。字符串直接为python对象，不需要item()
            splited_data=split_data(data,node,sonclass)
            mytree[node][sonclass]=creat_tree(splited_data)
    return mytree        

In [286]:
mytree=creat_tree(data)

[1.37744375 1.99999999] 1.3774437453109543 no surfacing
[-2.88539032e-09] -2.8853903190734774e-09 flippers


In [211]:
#use for learn #

s=pd.Series([1,5,3,3,3])
a=s.unique()
a.sort() #sort 原地修改
print(a)

s.unique().sort() is None

[1 3 5]


True

In [212]:
text_data=pd.DataFrame([[1,1,],
                   [0,1],
                   [1,0],
                   [0,0],
                   [1,1,]],columns=['no surfacing','flippers'])

In [213]:
mytree

{'no surfacing': {1: {'flippers': {1: 'yes', 0: 'no'}}, 0: 'no'}}

In [214]:
def classify(mytree,text_data_index):#apply(axis=)
    node=list(mytree.keys())[0]
    otherpart_tree=mytree[node]
    branch=list(otherpart_tree.keys())
    for i in branch:
        if i==text_data_index.loc[node]:
            if type(otherpart_tree[i])==dict:    #type(otherpart_tree[i])==dict type(otherpart_tree[i]).__name__='dict'
                data_class=classify(otherpart_tree[i],text_data_index)
            else:
                data_class=otherpart_tree[i]
    return data_class
    

In [215]:
s=text_data.iloc[0]
print(s)
classify(mytree,s)


no surfacing    1
flippers        1
Name: 0, dtype: int64


'yes'

In [216]:
text_data.apply(lambda x:classify(mytree,x),axis=1)

0    yes
1     no
2     no
3     no
4    yes
dtype: object

In [217]:
#text_data.apply(classify,axis=1,args=(mytree))  

#if we want to use the key 'arge=',the 'axis' will be the first key of the function.
#it's too unconvenient to change the sequence of the key,so there is no example. 

In [237]:
#将树保存到磁盘

In [144]:
def dump_tree(input_tree,filename):
    import pickle
    with open (filename,'wb') as f:
        pickle.dump(input_tree,f)
    return 0

In [145]:
dump_tree(mytree,'my_first_tree_classFish.pkl')

0

In [148]:
def load_tree(filename):
    import pickle
    with open (filename,'rb') as f:
        tree=pickle.load(f)
    print('grabbed tree')
    return tree

In [149]:
tree=load_tree('my_first_tree_classFish.pkl')

grabbed tree


In [150]:
tree

{'no surfacing': {1: {'flippers': {1: 'yes', 0: 'no'}}, 0: 'no'}}

In [151]:
#=======处理连续值=======#

In [218]:
s1=pd.DataFrame([[1,2,3],[4,5,6]])
s2=s1.iloc[:,0]
s3=s2
print(s2)
s3[0]=111
print(s2,'\n\n',s3) #s2 is changed.Because s3=s2 use the same value
#so we shoule use 'copy'
s4=s2.copy()
s4[0]=-99
print(s4,'\n\n',s2)

0    1
1    4
Name: 0, dtype: int64
0    111
1      4
Name: 0, dtype: int64 

 0    111
1      4
Name: 0, dtype: int64
0   -99
1     4
Name: 0, dtype: int64 

 0    111
1      4
Name: 0, dtype: int64


In [318]:
data2=pd.DataFrame([[1,'yes','fish'],
                   [3,'yes','fish'],
                   [2,'no','not'],
                   [6,'yes','not'],
                   [7,'yes','not']],columns=['no_surfacing','flippers','result'])

In [319]:
def information_entropy(data,P=None,N=None):
    data=data.set_index('result')
    length=len(data)
    
    try:
        positive=len(data.loc[P])
    except:
        positive=0
    try:
        nagetive=len(data.loc[N])
    except:
        nagetive=0    #in fact,the classify should stop
    ent=-(positive/length*np.log2(positive/length+1e-9)+nagetive/length*np.log2(nagetive/length+1e-9))
    return ent


In [320]:

def deal_continuous(data,column,data_name=None):
    data_forDeal=data[column]
    data_tem=data_forDeal.copy()    #the reason
    data_tem[-2:]=0
    
    data_two_mean=data_forDeal.sort_values().cumsum().values
#data_forDeal_index=data_forDeal.index
    data_tem=data_tem.sort_values().cumsum().values
#data_tem.index=data_forDeal_index
    data_two_mean=(data_two_mean-data_tem)/2
    data_two_mean=data_two_mean[1:]    #a little math knowladge

    ent=99
    data_len=len(data)
    sonclass=''
    for i in data_two_mean:
        class1=data.loc[data[column]>i]
        class1_len=len(class1)
        ent_1=class1_len/data_len*information_entropy(class1,P='fish',N='not')
        
        class2=data.loc[data[column]<=i]
        class2_len=len(class2)
        ent_2=class2_len/data_len*information_entropy(class2,P='fish',N='not')

        if (ent_1+ent_2)<ent:
            ent=ent_1+ent_2
            sonclass=f'{data_name}["{column}"]&{i}'
            expression=f'{column}&{i}'
    return (ent.item(),sonclass,expression)
        
#尚未考虑有多类连续值时
#用两遍得了

In [290]:
#到时候可能要用正则表达式处理？
#A:eval
ent1,condition,expression=deal_continuous(data2,'no_surfacing',data_name='data2')
deal_continuous(data2,'no_surfacing',data_name='data2')

(0.2490224969704552, 'data2["no_surfacing"]&1.5', 'no_surfacing&1.5')

In [321]:
def find_bestsonclass_continuous(data,**kwargs):    #kwargs:continuous_value=ent
                                                    #if there is ' ' in the column,we need to replace it with '_'
    min_ent=99
    bestsonclass=''
    if kwargs!={}:
        bestsonclass,min_ent=min(kwargs.items(),key=lambda x:x[1])
        
        continuous_columns=list(kwargs.keys())
        data_columns=list(data.columns)[:-1]
        data_columnclass=[x for x in data_columns if x not in continuous_columns]
        #data_columnclass=data.reindex(data_class_columns)
        
    else:
        data_columnclass=data.columns[:-1]    #remove the column of 'result' and get all the columns.
    data_len=len(data_columnclass)
    ent=np.zeros(data_len)
    #bestsonclass=''
    #min_ent=None
    _=0

    for i in data_columnclass:
        data_choose=data.set_index(i)
        sonclass_name=list(set(data_choose.index))
        sonclass_number=len(sonclass_name)

        for j in range(sonclass_number):
            sonclass=data_choose.loc[[sonclass_name[j]]] #use[[ ]] in case of becoming a Series
            each_sonclass_len=len(sonclass)
            ent[_]+=each_sonclass_len/data_len*information_entropy(sonclass,P='fish',N='not')    #信息增益(information gain) 
            
            #本来是求最大信息增益，这里算的是每一种分法的子类的信息熵之和，所以求最小。

        if min_ent==99:
            min_ent=ent[_]
            bestsonclass=i
        elif min_ent>ent[_]:
            min_ent=ent[_]
            bestsonclass=i
    
        _+=1

    print(ent,min_ent,bestsonclass)
    return bestsonclass

In [322]:
def split_data(data,name,value):
    data_split=data.set_index(name)
    #sonclass_values=list(set(data.index)).sort()
   
    return data_split.loc[[value]]    #离谱

In [323]:
def get_major_result_plas(data):
    data_tem=data.value_counts().sort_values()
    major_result=data_tem.index[0][-1]
    return major_result

In [328]:
lis=[]
def creat_tree_continuous(data,**kwargs):    #continuous_value=ent
    global lis
    lis.append(1)
    if len(data[['result']].value_counts())==1:
        return data['result'].iloc[0]   
        
    elif len(data.columns)==1 or len(data.iloc[:-1].value_counts())==1:
        return get_major_result_plas(data)
    #elif pd.isna(data.loc[:,:-2].values).all:   #NaN is not equal anything.Include itself.
    #    pass

    elif len(lis)>50:
        return -999
    
    else: 
        node=find_bestsonclass_continuous(data,**kwargs)    
        mytree={node:{}}

        if node in kwargs:
            global condition,expression
            for i in ['>','<=']:
                splited_data=data2[eval(condition.replace('&',i))]
                splited_data=splited_data.set_index(node)
                mytree[node][expression.replace('&',i)]=creat_tree_continuous(splited_data)
            
        else:
            sonclass_values=data[node].unique()
            for sonclass in sonclass_values:
                splited_data=split_data(data,node,sonclass)
                mytree[node][sonclass]=creat_tree_continuous(splited_data)
    return mytree        

In [330]:
ent1,condition,expression=deal_continuous(data2,'no_surfacing',data_name='data2')
creat_tree_continuous(data2,no_surfacing=ent1)

[3.99999999] 0.2490224969704552 no_surfacing
[1.169925] 1.1699249971142276 flippers


{'no_surfacing': {'no_surfacing>1.5': {'flippers': {'yes': 'fish',
    'no': 'not'}},
  'no_surfacing<=1.5': 'fish'}}

In [2]:
#不对！只实现了在第一次创建树的时候判断连续数据